# 🎤 Clone Your Own Voice (Free) — ClarityVoice

This notebook uses the free, open-source **XTTS-v2** model to make speech in **your** voice, from a short recording of you talking.

### How to use
1. **Turn on the free GPU:** menu **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**.
2. Run each cell **top to bottom** (click the ▶ on the left of each cell).
3. When asked, upload a **clear 15–30 second recording of your own voice** (WAV or MP3, no background noise — just you talking naturally).
4. Type what you want it to say, run the last cell, and it plays + downloads your cloned voice.

> First run downloads the model (~2 min). That's normal.

### 1. Check the GPU is on

In [ ]:
!nvidia-smi -L
# If this says 'command not found' or shows no GPU, go to Runtime > Change runtime type > T4 GPU.

### 2. Install the voice library (~1 min)

In [ ]:
!pip install -q coqui-tts "transformers==4.49.0"
print("Installed. Now do Runtime > Restart session, then run from cell 3 (skip this install).")

### 3. Load the model

In [ ]:
import os, torch
os.environ["COQUI_TOS_AGREED"] = "1"

# Make XTTS load on new PyTorch. Patch ONCE (safe to re-run — avoids RecursionError):
if not getattr(torch, "_xtts_patched", False):
    _real_torch_load = torch.load
    def _safe_load(*args, **kwargs):
        kwargs["weights_only"] = False
        return _real_torch_load(*args, **kwargs)
    torch.load = _safe_load
    torch._xtts_patched = True

from TTS.api import TTS
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("Model loaded ✅")

### 4. Upload YOUR voice recordings (multiple is better)
**Name each file with its language** — e.g. `english1.m4a`, `english2.m4a`, `hindi1.m4a`, `hindi2.m4a`.
For the best **accent**: upload English clips of you speaking English, and Hindi clips of you speaking Hindi. The notebook matches them automatically.

In [ ]:
from google.colab import files
print("Upload your recordings (hold Ctrl to pick multiple).")
print("Name them with the language: english1.m4a, english2.m4a, hindi1.m4a, hindi2.m4a")
uploaded = files.upload()
all_files = list(uploaded.keys())

# Match clips to language by file name, so English output uses YOUR English accent.
en_samples = [f for f in all_files if "eng" in f.lower() or f.lower().startswith("en")]
hi_samples = [f for f in all_files if "hin" in f.lower() or "urdu" in f.lower() or f.lower().startswith("hi")]
if not en_samples: en_samples = all_files   # fallback: use everything
if not hi_samples: hi_samples = all_files
ref = {"en": en_samples, "hi": hi_samples}

print("\nEnglish reference clips:", en_samples)
print("Hindi reference clips:  ", hi_samples)

### 5. Make it speak in your voice — multiple languages ✨
Each line below is one language. Edit the text, then run. This model supports **English (`en`)** and **Hindi (`hi`)**. Kannada is **not** supported here (ask for the Indic model for Kannada).

In [ ]:
from IPython.display import Audio, display

# Make it HUMAN through the text: fillers (umm, haa), stretched words (niiice),
# and pauses (commas, ...). The voice says exactly what you write — so write it human.
clips = {
    "en": "Umm... hey! So, this is actually my own voice, you know? Pretty niiice, right? Haha.",
    "hi": "अरे... हाँ! तो, ये सच में मेरी अपनी आवाज़ है, समझे? मज़ेदार है ना? हाहा।",
}

for lang, text in clips.items():
    out = f"my_voice_{lang}.wav"
    tts.tts_to_file(
        text=text, speaker_wav=ref[lang], language=lang, file_path=out,  # language-matched clips
        temperature=0.9,          # higher = more expressive (too high can add artifacts)
        repetition_penalty=4.0,
        length_penalty=1.0,
        top_k=50, top_p=0.9,
        speed=1.0,
        enable_text_splitting=True,
    )
    print(f"\n=== {lang} ===")
    display(Audio(out))
    files.download(out)

print("\nTip: the umm / haa / niiice / ... / ! in the text is what makes it sound human.")
print("Later, your AI assistant's 'brain' will write the text in this style automatically.")